# Week 4, Notebook 5: Predicting GDP

**How fast will a Caribbean economy grow this year?**

*Caribbean AI - Adrian Dunkley*

---

### What you will build
- A neural network that predicts a country's **GDP growth** (how much its economy
  grows in a year) from economic clues.
- A head-to-head with linear regression that shows **why** a network can win.
- A tour of twelve Caribbean economies, from tourism islands to oil producers.

### The big idea
**GDP** is the total value of everything a country produces in a year. Its growth
rate tells you if the economy is booming or shrinking. Different Caribbean
economies run on different engines: **Barbados, the Bahamas, and Saint Lucia**
live on tourism; **Trinidad and Tobago, Guyana, and Suriname** on oil; **Haiti**
leans heavily on money sent home by relatives abroad (remittances).

That variety is exactly why a neural network helps: the effect of one clue
*depends on another*. A pandemic barely dents an oil economy but devastates a
tourism one. A straight-line model cannot bend like that. A network can.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

gdp = pd.read_csv("data/caribbean_gdp.csv")
print("Rows (country-years):", len(gdp))
print("Countries:", gdp["country"].nunique())
gdp.head()

### The clues we use

| Column | What it means |
| --- | --- |
| `tourism_reliance` | Share of the economy from tourism (0 to 1). |
| `remittance_reliance` | Share from money sent home by relatives abroad. |
| `is_oil_exporter` | 1 if the country sells oil (Trinidad, Guyana, Suriname). |
| `tourism_arrivals_growth` | Change in tourist numbers this year (%). |
| `oil_price_usd` | The world oil price this year. |
| `remittance_growth` | Change in remittances this year (%). |
| `inflation` | How fast prices rose. |
| `world_growth` | How fast the whole world's economy grew. |
| `crisis_year` | 1 in 2009 (financial crash) and 2020 (pandemic). |
| `pandemic_year` | 1 in 2020 only. |
| `gdp_growth` | What we predict: the country's GDP growth that year (%). |

In [ ]:
# The pandemic hit tourism economies far harder than oil ones.
# This is the "it depends" pattern a straight line struggles with.
pandemic = gdp[gdp["year"] == 2020].sort_values("gdp_growth")
plt.figure(figsize=(9, 5))
colors = ["#c0392b" if r > 0.3 else "#2980b9" for r in pandemic["tourism_reliance"]]
plt.barh(pandemic["country"], pandemic["gdp_growth"], color=colors)
plt.title("2020 pandemic: GDP growth by country\n(red = tourism-reliant, blue = less so)")
plt.xlabel("GDP growth (%)"); plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

### Step 1: Inputs, split, and scale

Here we predict across many countries and years, so a **random** split is fine
(each country-year is its own example). We scale as always.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features = ["tourism_reliance", "remittance_reliance", "is_oil_exporter",
            "tourism_arrivals_growth", "oil_price_usd", "remittance_growth",
            "inflation", "world_growth", "crisis_year", "pandemic_year"]
X = gdp[features].values
y = gdp["gdp_growth"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
print("Train:", len(X_train), " Test:", len(X_test))

### Step 2: Linear regression first

Start simple. A linear model assumes each clue adds a fixed amount to growth, no
matter what the other clues say. Let us see how far that gets us.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

linreg = LinearRegression().fit(X_train_s, y_train)
lin_pred = linreg.predict(X_test_s)
print(f"Linear model R2:  {r2_score(y_test, lin_pred):.3f}")
print(f"Linear model MAE: {mean_absolute_error(y_test, lin_pred):.2f} percentage points")

### Step 3: The neural network

Now the network. Its hidden layers let it learn **interactions**, like "a
pandemic *times* high tourism reliance means a deep crash". That is the kind of
bending a straight line cannot do.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(3)

model = keras.Sequential([
    layers.Input(shape=(len(features),)),
    layers.Dense(48, activation="relu"),
    layers.Dense(24, activation="relu"),
    layers.Dense(1),
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = model.fit(X_train_s, y_train, validation_split=0.15,
                    epochs=300, batch_size=16, verbose=0)

plt.figure(figsize=(7, 4))
plt.plot(history.history["mae"], label="training")
plt.plot(history.history["val_mae"], label="validation")
plt.title("Learning curve (MAE)"); plt.xlabel("epoch"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
nn_pred = model.predict(X_test_s, verbose=0).flatten()
print("--- Test set (unseen country-years) ---")
print(f"Linear   R2 = {r2_score(y_test, lin_pred):.3f}   MAE = {mean_absolute_error(y_test, lin_pred):.2f}")
print(f"Neural   R2 = {r2_score(y_test, nn_pred):.3f}   MAE = {mean_absolute_error(y_test, nn_pred):.2f}")
print("\nThe network wins - because it learns the 'it depends' interactions.")

In [ ]:
# How close are predictions to reality? Points near the dashed line are good.
plt.figure(figsize=(6, 6))
plt.scatter(y_test, nn_pred, alpha=0.6)
lo, hi = y_test.min(), y_test.max()
plt.plot([lo, hi], [lo, hi], "k--", label="perfect prediction")
plt.title("Predicted vs actual GDP growth")
plt.xlabel("actual growth (%)"); plt.ylabel("predicted growth (%)")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

### Step 4: Ask the model a "what if" question

Because the network learned how tourism and shocks interact, we can pose
scenarios. Compare a tourism-heavy island (like the Bahamas) with an oil
exporter (like Trinidad) in a **normal** year and a **pandemic** year.

In [ ]:
def scenario(tourism, remit, oil_exp, tourism_growth, oil_price,
             remit_growth, inflation, world_growth, crisis, pandemic):
    row = np.array([[tourism, remit, oil_exp, tourism_growth, oil_price,
                     remit_growth, inflation, world_growth, crisis, pandemic]])
    return model.predict(scaler.transform(row), verbose=0)[0, 0]

print("Predicted GDP growth:\n")
print("Bahamas-like (tourism 0.50), normal year: "
      f"{scenario(0.50, 0.02, 0, 4, 60, 2, 3, 3, 0, 0):+.1f}%")
print("Bahamas-like (tourism 0.50), PANDEMIC:     "
      f"{scenario(0.50, 0.02, 0, -60, 40, -5, 2, -4, 1, 1):+.1f}%")
print("Trinidad-like (oil exporter), normal year: "
      f"{scenario(0.08, 0.03, 1, 3, 80, 2, 4, 3, 0, 0):+.1f}%")
print("Trinidad-like (oil exporter), PANDEMIC:    "
      f"{scenario(0.08, 0.03, 1, -20, 40, -3, 3, -4, 1, 1):+.1f}%")

### What you learned

- **GDP growth** can be predicted from economic clues, and Caribbean economies
  differ sharply by their main engine: tourism, oil, or remittances.
- A neural network **beats linear regression** here because it learns
  **interactions** ("a pandemic hurts tourism economies most"), which a straight
  line cannot capture.
- A trained model can answer **"what if"** questions, a powerful tool for
  planners and economists.

### Try it yourself
1. Drop `pandemic_year` from the features and retrain. Watch the network's
   advantage over the linear model shrink.
2. Run a scenario for a remittance-reliant economy (like Haiti, remittance
   reliance 0.35) during a global slowdown.
3. Which single country-year is the model most wrong about? (Hint: sort by the
   size of the error.)